# Task 3: Filter Invalid Records Safely

## Step 1: Initialize & Load All Data
Include header and empty lines for full picture.

In [1]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

raw_rdd = sc.textFile("/home/jovyan/data/raw/employees.txt")

print(f"Total raw lines: {raw_rdd.count()}")

Total raw lines: 13


## Step 2: Identify Issue Types

We have 3 issues:
- **Header row** (line 1)
- **Empty lines** (lines 3, 9)
- **Malformed record** (line 7: missing comma)

In [2]:
header = raw_rdd.first()

# Tag each line with its type
def classify_line(line):
    if line == header:
        return ("header", line)
    elif line.strip() == "":
        return ("empty", line)
    else:
        parts = line.split(",")
        if len(parts) != 9:
            return ("malformed", line)
        return ("valid", line)

classified = raw_rdd.map(classify_line)

# Count by type
type_counts = classified.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)

print("Line classification:")
for line_type, count in type_counts.collect():
    print(f"  {line_type:<12} → {count}")

Line classification:
  header       → 1
  empty        → 2
  valid        → 10


## Step 3: Extract Only Valid Records
Filter out header, empty lines, and malformed records.

In [3]:
valid_only = classified.filter(lambda x: x[0] == "valid").map(lambda x: x[1])

print(f" Valid records: {valid_only.count()}")

 Valid records: 10


## Step 4: Safe Parsing with Validation
Parse valid lines into structured records with type checks.

In [4]:
def safe_parse(line):
    """Parse line. Returns (True, dict) if valid, else (False, error_msg)."""
    parts = line.split(",")
    
    if len(parts) != 9:
        return (False, f"Field count: {len(parts)}")
    
    try:
        record = {
            "emp_id": int(parts[0]),
            "name": parts[1],
            "department": parts[2],
            "job_title": parts[3],
            "salary": float(parts[4]),
            "location": parts[5],
            "hire_date": parts[6],
            "performance_rating": float(parts[7]),
            "years_exp": int(parts[8].strip())
        }
        return (True, record)
    except (ValueError, IndexError) as e:
        return (False, str(e))

# Apply to valid lines
parsed = valid_only.map(safe_parse)
parsed.cache()

success = parsed.filter(lambda x: x[0]).map(lambda x: x[1])
failed = parsed.filter(lambda x: not x[0]).map(lambda x: x[1])

print(f"Parsed: {success.count()}")
print(f"Failed: {failed.count()}")

Parsed: 9
Failed: 1


## Step 5: Inspect Malformed Record
Understand what went wrong.

In [5]:
# Show records that passed field-count check but failed type validation
failed_records = parsed.filter(lambda x: not x[0]).map(lambda x: x[1])

print("Records that passed field count but failed type check:")
for err in failed_records.collect():
    print(f"  Error: {err}")

# Also show the raw line that caused the issue
print("\n Root cause — raw line with missing comma:")
raw_line = valid_only.filter(lambda x: "Legal Counsel" in x).first()
print(f"  {raw_line}")
print(f"  After split: {raw_line.split(',')}")
print(f"  'Legal Counsel145000' should be 'Legal Counsel,145000'")

Records that passed field count but failed type check:
  Error: could not convert string to float: 'San Francisco'

 Root cause — raw line with missing comma:
  7,Robert Martinez,Legal,Legal Counsel145000,San Francisco,2019-09-22,4.8,10, 5
  After split: ['7', 'Robert Martinez', 'Legal', 'Legal Counsel145000', 'San Francisco', '2019-09-22', '4.8', '10', ' 5']
  'Legal Counsel145000' should be 'Legal Counsel,145000'


## Step 6: Display Final Clean Records

In [ ]:
print("Clean structured records:\n")
for rec in success.collect():
    print(f"  [{rec['emp_id']}] {rec['name']:<20} | {rec['department']:<12} | "
          f"{rec['job_title']:<18} | ${rec['salary']:>8,.0f} | "
          f"Rating: {rec['performance_rating']}")

Clean structured records:

  [1] John Smith           | Engineering  | Senior Developer   | $ 125,000 | Rating: 4.5
  [2] Sarah Johnson        | Sales        | Account Executive  | $  85,000 | Rating: 4.2
  [3] Michael Williams     | Engineering  | Software Engineer  | $  95,000 | Rating: 3.8
  [4] Jennifer Brown       | Marketing    | Marketing Manager  | $  92,000 | Rating: 4.7
  [5] David Jones          | Finance      | Senior Analyst     | $ 105,000 | Rating: 4.3
  [6] Lisa Garcia          | IT           | DevOps Engineer    | $ 115,000 | Rating: 4.6
  [8] Patricia Wilson      | HR           | HR Manager         | $  88,000 | Rating: 4.1
  [9] James Anderson       | Sales        | Sales Manager      | $ 110,000 | Rating: 4.4
  [10] Mary Thomas          | Engineering  | Tech Lead          | $ 145,000 | Rating: 4.9


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 38960)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

## Task 3 Complete

| Category | Count | Action |
|----------|-------|--------|
| Header | 1 | Skipped |
| Empty lines | 2 | Skipped |
| Malformed | 1 | Logged & skipped |
| Valid parsed | 9 | Ready for analysis |

## Task 3 Complete

**Strategy:** Two-layer validation (field count → type check).

**Result:** 9 clean records, 1 malformed safely excluded.